In [1]:
import chipwhisperer as cw
import time, os
from Crypto.Cipher import AES
bitstream_path =  r'/home/sareeta/chipwhisperer/firmware/fpgas/aes/vivado/cw305_aes.runs/impl_100t/cw305_top.bit'
assert os.path.isfile(bitstream_path), f"Bitstream not found: {bitstream_path}"

# 2) Connect to the capture board (CWLite)
scope = cw.scope()
scope.default_setup()

# 3) Connect and Program the FPGA
print("Programming CW305 FPGA with:", bitstream_path)
target = cw.target(scope, cw.targets.CW305, bsfile=bitstream_path, force=True)

(ChipWhisperer NAEUSB WARNING|File naeusb.py:826) Your firmware (0.51.0) is outdated - latest is 0.54.0 See https://chipwhisperer.readthedocs.io/en/latest/firmware.html for more information


scope.gain.mode                          changed from low                       to high                     
scope.gain.gain                          changed from 0                         to 30                       
scope.gain.db                            changed from 5.5                       to 24.8359375               
scope.adc.basic_mode                     changed from low                       to rising_edge              
scope.adc.samples                        changed from 24400                     to 5000                     
scope.adc.trig_count                     changed from 2022427                   to 24910682                 
scope.clock.adc_src                      changed from clkgen_x1                 to clkgen_x4                
scope.clock.adc_freq                     changed from 0                         to 29538459                 
scope.clock.adc_rate                     changed from 0.0                       to 29538459.0               
scope.clock.freq_ct

(ChipWhisperer Target WARNING|File CW305.py:591) Using default Verilog defines (/home/sareeta/chipwhisperer/software/chipwhisperer/hardware/firmware/cw305/cw305_aes_defines.v); if this is not what you want, provide them via the defines_files argument


In [2]:
# Fixed test vector
KEY = bytes.fromhex("000102030405060708090a0b0c0d0e0f")
PT  = bytes.fromhex("00112233445566778899aabbccddeeff")

# Software AES (ground truth)
sw_ct = AES.new(KEY, AES.MODE_ECB).encrypt(PT)

# Send to FPGA
target.fpga_write(target.REG_CRYPT_KEY, KEY)
target.fpga_write(target.REG_CRYPT_TEXTIN, PT)
target.fpga_write(target.REG_CRYPT_GO, b"\x01")
time.sleep(0.01)

hw_ct = bytes(target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16))

print("=== NORMAL AES OPERATION ===")
print("Plaintext :", PT.hex())
print("Key       :", KEY.hex())
print("SW AES CT :", sw_ct.hex())
print("HW AES CT :", hw_ct.hex())
print("MATCH?    :", sw_ct == hw_ct)


=== NORMAL AES OPERATION ===
Plaintext : 00112233445566778899aabbccddeeff
Key       : 000102030405060708090a0b0c0d0e0f
SW AES CT : 69c4e0d86a7b0430d8cdb78070b4c55a
HW AES CT : 69c4e0d86a7b0430d8cdb78070b4c55a
MATCH?    : True


In [3]:
print(scope.trigger.triggers)
scope.trigger.triggers = "tio4"

tio4


In [4]:
print(scope.glitch.clk_src)
print(scope.clock.clkgen_freq)
print(scope.clock.adc_freq)

target
7384615.384615385
29538459


In [11]:
scope.dis()

(ChipWhisperer Scope ERROR|File naeusbchip.py:113) Scope already disconnected!


True

In [5]:
def aes_encrypt_once():
    # fire AES
    target.fpga_write(target.REG_CRYPT_TEXTIN, PT)
    target.fpga_write(target.REG_CRYPT_GO, b"\x01")
    # small wait so ciphertext register updates even if trigger missed
    time.sleep(0.001)
    return bytes(target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16))

def glitch_and_read():
    # arm scope + glitch
    scope.arm()

    # launch AES (this generates the external trigger used by ext_single)
    target.fpga_write(target.REG_CRYPT_TEXTIN, PT)
    target.fpga_write(target.REG_CRYPT_GO, b"\x01")

    # capture waveform (optional but useful for debugging alignment)
    cap_timeout = scope.capture()
    ct = bytes(target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16))

    if cap_timeout:
        return "no_trigger", ct, None

    tr = scope.get_last_trace()

    if ct == hw_ct:
        return "correct", ct, tr
    else:
        return "faulty", ct, tr


In [4]:
scope.glitch.clk_src = "clkgen"
scope.glitch.output = "glitch_only"
scope.glitch.repeat = 1
scope.io.hs2 = "glitch"
scope.io.glitch_lp = True
scope.io.glitch_hp = False
scope.glitch.clk_src = 'clkgen'
scope.trigger.triggers = "tio4" 
scope.glitch.trigger_src ="ext_single" #"continuous" #change this depending on glitching desired   "ext_single"  "continuous"  
print("Glitch ready.")
print("voltage glitch engine armed (but harmless so far)")

Glitch ready.
voltage glitch engine armed (but harmless so far)


In [8]:
scope.glitch.clk_src = "clkgen"
scope.glitch.output = "glitch_only"
scope.glitch.repeat = 1
scope.io.glitch_lp = False
scope.io.glitch_hp = True
# scope.gain.gain = 46
# scope.gain.mode = "high"
# scope.adc.samples = 500
# scope.adc.offset = 0
# scope.adc.basic_mode = "rising_edge"
# scope.clock.adc_src = "clkgen_x4"
#scope.clock.freq_ctr_src = "clkgen"
scope.clock.adc_phase = 0
scope.trigger.triggers = "tio4"
scope.clock.extclk_freq = 10000000
scope.clock.clkgen_mul = 5
scope.clock.clkgen_div = 48
scope.clock.clkgen_freq = 10000000
scope.io.hs2 = "clkgen"
scope.glitch.clk_src = 'clkgen'
scope.glitch.ext_offset = 0
scope.glitch.trigger_src ="ext_single" #"continuous" #change this depending on glitching desired   "ext_single"  "continuous"
#scope.glitch.output = "clock_xor"
scope.glitch.width = 10
scope.glitch.offset = -20
#self.api.setParameter(['Glitch Module', 'Output Mode', 'Clock XORd'])
scope.glitch.repeat = 5   
print("Glitch ready.")
print("voltage glitch engine armed (but harmless so far)")

Glitch ready.
voltage glitch engine armed (but harmless so far)


In [9]:
import numpy as np

# Configuration
N_PER_SETTING = 5
records = []
traces = []
labels = []
faults = []
hits = {"correct": 0, "faulty": 0, "no_trigger": 0}

# Range Setup
ext_offsets = range(0, 1, 1)
widths = [x for x in range(-49, 1, 1) if x != 0]
offsets = [x for x in range(-49, 49, 1) if x != 0]
#widths  = range(-49, 1, 1)
#offsets = range(-49, 49, 1)

print("Starting Glitch Loop...")
for ext in ext_offsets:
    scope.glitch.ext_offset = ext
    for w in widths:
        scope.glitch.width = w
        for off in offsets:
            scope.glitch.offset = off
            
            for rep in range(N_PER_SETTING):
                # --- THE WORKING LOGIC FROM CODE 1 ---
                scope.arm()
                
                # Launch AES
                target.fpga_write(target.REG_CRYPT_TEXTIN, PT)
                target.fpga_write(target.REG_CRYPT_GO, b"\x01")
                
                # Capture the hardware result
                cap_timeout = scope.capture()
                ct = bytes(target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16))
                
                # Trace processing (from Code 2)
                tr = None
                if not cap_timeout:
                    tr = np.array(scope.get_last_trace(), dtype=np.float32)
                # --------------------------------------
    
                # Labeling Logic
                if cap_timeout:
                    label = "no_trigger"
                elif ct == hw_ct:
                    label = "correct"
                else:
                    label = "faulty"
                    faults.append((w, off, ct.hex()))
                    print(f"FAULT! w={w} off={off} ct={ct.hex()}")
    
                # Store in Hits dictionary
                hits[label] += 1
    
                # Create the record (from Code 2)
                records.append({
                    "ext":ext,
                    "width": w,
                    "offset": off,
                    "rep": rep,
                    "label": label,
                    "ct_hex": ct.hex(),
                })
    
                # Store trace if it exists
                if tr is not None:
                    traces.append(tr)
                    labels.append(label)

print("\n--- Scan Complete ---")
print("Summary:", hits)
print("Total records:", len(records))

Starting Glitch Loop...

--- Scan Complete ---
Summary: {'correct': 23765, 'faulty': 0, 'no_trigger': 0}
Total records: 23765


In [4]:
import numpy as np

# Configuration
N_PER_SETTING = 5
records = []
traces = []
labels = []
faults = []
hits = {"correct": 0, "faulty": 0, "no_trigger": 0}

# Range Setup
widths = [x for x in range(-49, 1, 1) if x != 0]
offsets = [x for x in range(-49, 49, 1) if x != 0]

print("Starting Glitch Loop...")

for w in widths:
    scope.glitch.width = w
    for off in offsets:
        scope.glitch.offset = off
        
        for rep in range(N_PER_SETTING):
            # --- THE WORKING LOGIC FROM CODE 1 ---
            scope.arm()
            
            # Launch AES
            target.fpga_write(target.REG_CRYPT_TEXTIN, PT)
            target.fpga_write(target.REG_CRYPT_GO, b"\x01")
            
            # Capture the hardware result
            cap_timeout = scope.capture()
            ct = bytes(target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16))
            
            # Trace processing (from Code 2)
            tr = None
            if not cap_timeout:
                tr = np.array(scope.get_last_trace(), dtype=np.float32)
            # --------------------------------------

            # Labeling Logic
            if cap_timeout:
                label = "no_trigger"
            elif ct == hw_ct:
                label = "correct"
            else:
                label = "faulty"
                faults.append((w, off, ct.hex()))
                print(f"FAULT! w={w} off={off} ct={ct.hex()}")

            # Store in Hits dictionary
            hits[label] += 1

            # Create the record (from Code 2)
            records.append({
                "width": w,
                "offset": off,
                "rep": rep,
                "label": label,
                "ct_hex": ct.hex(),
            })

            # Store trace if it exists
            if tr is not None:
                traces.append(tr)
                labels.append(label)

print("\n--- Scan Complete ---")
print("Summary:", hits)
print("Total records:", len(records))

Starting Glitch Loop...
FAULT! w=1 off=-49 ct=ffffffffffffffffffffffffffffffff


(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:730) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 0b
(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:730) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 0a
(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:730) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 0b
(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:730) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 0a
(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:730) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 0b
(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:730) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 0a
(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:730) Timeout in OpenADC capture(), no t

USBErrorNoDevice: LIBUSB_ERROR_NO_DEVICE [-4]

In [5]:
import numpy as np

# Configuration
N_PER_SETTING = 5
records = []
traces = []
labels = []
faults = []
hits = {"correct": 0, "faulty": 0, "no_trigger": 0}

# Range Setup
ext_offsets = range(0,5,1)
widths = [x for x in range(-49, 49, 1) if x != 0]
offsets = [x for x in range(-49, 49, 1) if x != 0]

print("Starting Glitch Loop...")
for ext in ext_offsets:
    scope.glitch.ext_offset = ext
    for w in widths:
        scope.glitch.width = w
        for off in offsets:
            scope.glitch.offset = off
            
            for rep in range(N_PER_SETTING):
                # --- THE WORKING LOGIC FROM CODE 1 ---
                scope.arm()
                
                # Launch AES
                target.fpga_write(target.REG_CRYPT_TEXTIN, PT)
                target.fpga_write(target.REG_CRYPT_GO, b"\x01")
                
                # Capture the hardware result
                cap_timeout = scope.capture()
                ct = bytes(target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16))
                
                # Trace processing (from Code 2)
                tr = None
                if not cap_timeout:
                    tr = np.array(scope.get_last_trace(), dtype=np.float32)
                # --------------------------------------
    
                # Labeling Logic
                if cap_timeout:
                    label = "no_trigger"
                elif ct == hw_ct:
                    label = "correct"
                else:
                    label = "faulty"
                    faults.append((w, off, ct.hex()))
                    print(f"FAULT! w={w} off={off} ct={ct.hex()}")
    
                # Store in Hits dictionary
                hits[label] += 1
    
                # Create the record (from Code 2)
                records.append({
                    "ext":ext,
                    "width": w,
                    "offset": off,
                    "rep": rep,
                    "label": label,
                    "ct_hex": ct.hex(),
                })
    
                # Store trace if it exists
                if tr is not None:
                    traces.append(tr)
                    labels.append(label)

print("\n--- Scan Complete ---")
print("Summary:", hits)
print("Total records:", len(records))

Starting Glitch Loop...


USBErrorIO: LIBUSB_ERROR_IO [-1]

In [4]:
scope.glitch.clk_src = "clkgen" 
scope.glitch.output = "clock_xor" 
scope.glitch.repeat = 1
#scope.io.glitch_hp = False 
# scope.gain.gain = 46 
# scope.gain.mode = "high" 
# scope.adc.samples = 500 
# scope.adc.offset = 0 
# scope.adc.basic_mode = "rising_edge" 
# scope.clock.adc_src = "clkgen_x4" 
scope.clock.freq_ctr_src = "clkgen" 
scope.clock.adc_phase = 0 
scope.trigger.triggers = "tio4" 
scope.clock.extclk_freq = 10000000 
scope.clock.clkgen_mul = 5 
scope.clock.clkgen_div = 48 
scope.clock.clkgen_freq = 10000000 
scope.io.hs2 = "glitch" 
scope.glitch.clk_src = 'clkgen' 
scope.glitch.ext_offset = 0 
scope.glitch.trigger_src ="ext_single" #"continuous" #change this depending on glitching desired "ext_single" "continuous" 
scope.glitch.output = "clock_xor" 
scope.glitch.width = 10 
scope.glitch.offset = -20 
#self.api.setParameter(['Glitch Module', 'Output Mode', 'Clock XORd']) 
scope.glitch.repeat = 5 
print("Glitch ready.") 
print("Clock glitch engine armed (but harmless so far)")

Glitch ready.
Clock glitch engine armed (but harmless so far)


In [ ]:
1
#scope.io.glitch_hp = False 
# scope.gain.gain = 46 
# scope.gain.mode = "high" 
# scope.adc.samples = 500 
# scope.adc.offset = 0 
# scope.adc.basic_mode = "rising_edge" 
# scope.clock.adc_src = "clkgen_x4" 
scope.clock.freq_ctr_src = "clkgen" 
scope.clock.adc_phase = 0 
scope.trigger.triggers = "tio4" 
scope.clock.extclk_freq = 10000000 
scope.clock.clkgen_mul = 5 
scope.clock.clkgen_div = 48 
scope.clock.clkgen_freq = 10000000 
scope.io.hs2 = "glitch" 
scope.glitch.ext_offset = 0 


In [4]:
scope.io.hs2 = "glitch" 
#scope.trigger.triggers = "tio4" 
scope.glitch.clk_src = 'clkgen' 
scope.glitch.trigger_src ="ext_single" #"continuous" #change this depending on glitching desired "ext_single" "continuous" 
scope.glitch.output = "clock_xor" 
scope.glitch.repeat = 5 
print("Glitch ready.") 
print("Clock glitch engine armed (but harmless so far)")

Glitch ready.
Clock glitch engine armed (but harmless so far)


In [5]:
import numpy as np

# Configuration
N_PER_SETTING = 5
records = []
traces = []
labels = []
faults = []
hits = {"correct": 0, "faulty": 0, "no_trigger": 0}

# Range Setup
widths = [x for x in range(-49, 8, 1) if x != 0]
offsets = [x for x in range(-49, 49, 1) if x != 0]

print("Starting Glitch Loop...")

for w in widths:
    scope.glitch.width = w
    for off in offsets:
        scope.glitch.offset = off
        
        for rep in range(N_PER_SETTING):
            # --- THE WORKING LOGIC FROM CODE 1 ---
            scope.arm()
            
            # Launch AES
            target.fpga_write(target.REG_CRYPT_TEXTIN, PT)
            target.fpga_write(target.REG_CRYPT_GO, b"\x01")
            
            # Capture the hardware result
            cap_timeout = scope.capture()
            ct = bytes(target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16))
            
            # Trace processing (from Code 2)
            tr = None
            if not cap_timeout:
                tr = np.array(scope.get_last_trace(), dtype=np.float32)
            # --------------------------------------

            # Labeling Logic
            if cap_timeout:
                label = "no_trigger"
            elif ct == hw_ct:
                label = "correct"
            else:
                label = "faulty"
                faults.append((w, off, ct.hex()))
                print(f"FAULT! w={w} off={off} ct={ct.hex()}")

            # Store in Hits dictionary
            hits[label] += 1

            # Create the record (from Code 2)
            records.append({
                "width": w,
                "offset": off,
                "rep": rep,
                "label": label,
                "ct_hex": ct.hex(),
            })

            # Store trace if it exists
            if tr is not None:
                traces.append(tr)
                labels.append(label)

print("\n--- Scan Complete ---")
print("Summary:", hits)
print("Total records:", len(records))

Starting Glitch Loop...
FAULT! w=-49 off=-49 ct=699c3bd86a7a0430d8ccb58070b5c75a
FAULT! w=-47 off=-49 ct=69c4e0d86a130420d845a60060b40412
FAULT! w=-46 off=-49 ct=938a093b40320aaae3e0a5bcb2abf3a1
FAULT! w=-46 off=46 ct=9e4c21c830852b5deb6b0fa1a30addee
FAULT! w=-45 off=-49 ct=1b76088c84c29fd9d3b33862be82fc1d
FAULT! w=-35 off=-49 ct=621ac859f2e7fbfc05ce7d0189a1f422
FAULT! w=-34 off=-49 ct=06c4c018ea7a72b0d8fc1700f0b47b5a
FAULT! w=-27 off=-49 ct=65c4e0d8a57b0464dccddc807421c55a
FAULT! w=-27 off=-49 ct=6dc43e4d6a7b04b3d8cdb71ba1b4935a
FAULT! w=-26 off=-49 ct=1c93fa7056d03985e7c4477c4e6c66c1
FAULT! w=-19 off=-29 ct=101340143fac2d404db3f987ce4c5d24
FAULT! w=-16 off=1 ct=a61ea7583d452e67203416d2d1905025
FAULT! w=-11 off=-39 ct=69c4e0c86a330430d8c5b78070a4c552
FAULT! w=-6 off=48 ct=3c0a9170e70ce78b694de8d11eea5759
FAULT! w=-5 off=5 ct=7f91482b8ab060b7246bdc6d449b96de
FAULT! w=-2 off=-49 ct=f1d2a9970cc1c30f6f2ecdb70b85b8da
FAULT! w=-1 off=-47 ct=8d199488779a8a71d799046bf257850f
FAULT! w=-1 off=-

In [6]:
import numpy as np

# Configuration
N_PER_SETTING = 5
records = []
traces = []
labels = []
faults = []
hits = {"correct": 0, "faulty": 0, "no_trigger": 0}

# Range Setup
ext_offsets = range(0, 5)   # 1 to 100
widths = range(-49, 49, 1)
offsets = range(-49, 49, 1)

print("Starting Glitch Loop...")

for ext_off in ext_offsets:
    scope.glitch.ext_offset = ext_off

    for w in widths:
        scope.glitch.width = w

        for off in offsets:
            scope.glitch.offset = off

            for rep in range(N_PER_SETTING):

                scope.arm()

                target.fpga_write(target.REG_CRYPT_TEXTIN, PT)
                target.fpga_write(target.REG_CRYPT_GO, b"\x01")

                cap_timeout = scope.capture()
                ct = bytes(target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16))

                tr = None
                if not cap_timeout:
                    tr = np.array(scope.get_last_trace(), dtype=np.float32)

                if cap_timeout:
                    label = "no_trigger"

                elif ct == hw_ct:
                    label = "correct"

                else:
                    label = "faulty"
                    faults.append((ext_off, w, off, ct.hex()))
                    print(
                        f"FAULT! ext={ext_off} w={w} off={off} ct={ct.hex()}"
                    )

                hits[label] += 1

                records.append({
                    "ext_offset": ext_off,
                    "width": w,
                    "offset": off,
                    "rep": rep,
                    "label": label,
                    "ct_hex": ct.hex(),
                })

                if tr is not None:
                    traces.append(tr)
                    labels.append(label)

print("\n--- Scan Complete ---")
print("Summary:", hits)
print("Total records:", len(records))

Starting Glitch Loop...


(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work


FAULT! ext=0 w=-46 off=48 ct=a6e78818db878d158092661bb7a445d5


(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work


FAULT! ext=0 w=-45 off=47 ct=3c0a9170e70ce78b694de8d11eea5759


...

FAULT! ext=2 w=1 off=-2 ct=61c4e0d8405a2418c0c4b4807094c55a
FAULT! ext=2 w=1 off=-2 ct=60e810802028202000a0000420a0c040
FAULT! ext=2 w=1 off=-2 ct=40e4f09000c8e420007894807094c452


(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work


FAULT! ext=2 w=5 off=45 ct=4840804040100020c044020020200010


(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfigu


--- Scan Complete ---
Summary: {'correct': 240030, 'faulty': 70, 'no_trigger': 0}
Total records: 240100


In [ ]:
from collections import Counter, defaultdict
import pandas as pd
import numpy as np

# Correct ciphertext reference
correct_ct = hw_ct  # bytes, length 16

def xor_bytes(a, b):
    return bytes(x ^ y for x, y in zip(a, b))

def changed_positions(diff_bytes):
    return tuple(i for i, v in enumerate(diff_bytes) if v != 0)

# -----------------------------
# Step 2: compute mask/diff
# -----------------------------
analysis_rows = []

for r in records:
    ct = bytes.fromhex(r["ct_hex"])

    diff = xor_bytes(correct_ct, ct)
    changed = changed_positions(diff)

    row = dict(r)
    row["diff_hex"] = diff.hex()
    row["changed_positions"] = changed
    row["num_changed_bytes"] = len(changed)
    row["pattern"] = ",".join(map(str, changed)) if changed else "none"

    analysis_rows.append(row)

df = pd.DataFrame(analysis_rows)

# Only faulty ciphertexts
fault_df = df[df["label"] == "faulty"].copy()

print("Total faulty:", len(fault_df))
print(fault_df[["ext", "width", "offset", "rep", "ct_hex", "diff_hex",
                "num_changed_bytes", "pattern"]].head(20))

In [ ]:
# -----------------------------
# Step 3: cluster by pattern
# -----------------------------
pattern_summary = (
    fault_df
    .groupby(["pattern", "num_changed_bytes"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

print("\nMost repeated fault patterns:")
print(pattern_summary.head(30).to_string(index=False))

In [ ]:
# -----------------------------
# Step 4: repeated pattern per setting
# -----------------------------
setting_pattern_summary = (
    fault_df
    .groupby(["ext", "width", "offset", "pattern", "num_changed_bytes"])
    .size()
    .reset_index(name="count")
    .sort_values(["count", "num_changed_bytes"], ascending=[False, True])
)

print("\nBest repeated patterns per setting:")
print(setting_pattern_summary.head(50).to_string(index=False))

In [ ]:
# -----------------------------
# AES-DFA candidate patterns
# -----------------------------
dfa_candidates = setting_pattern_summary[
    setting_pattern_summary["num_changed_bytes"].isin([1, 2, 3, 4])
].copy()

print("\nPotential DFA-useful candidates:")
print(dfa_candidates.head(50).to_string(index=False))

In [ ]:
df.to_csv("all_glitch_records_with_diff.csv", index=False)
fault_df.to_csv("faults_only_with_diff.csv", index=False)
pattern_summary.to_csv("pattern_summary.csv", index=False)
setting_pattern_summary.to_csv("setting_pattern_summary.csv", index=False)

print("Saved CSV files.")

In [9]:
candidates = [
    (1, -3),
    (1, 1),
    (3, -1),
    (-39, 1),
]

In [10]:
from collections import Counter

def test_setting(w, off, ext_off=0, n=500):
    cts = []
    timeouts = 0

    scope.glitch.ext_offset = ext_off
    scope.glitch.width = w
    scope.glitch.offset = off
    scope.glitch.repeat = 1

    for i in range(n):
        scope.arm()

        target.fpga_write(target.REG_CRYPT_TEXTIN, PT)
        target.fpga_write(target.REG_CRYPT_GO, b"\x01")

        timeout = scope.capture()

        if timeout:
            timeouts += 1
            continue

        ct = bytes(target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16)).hex()
        cts.append(ct)

    counts = Counter(cts)

    print("\nSETTING:", "w=", w, "off=", off, "ext=", ext_off)
    print("Timeouts:", timeouts)
    print("Correct:", counts[hw_ct.hex()])
    print("Faulty:", len(cts) - counts[hw_ct.hex()])
    print("Top 10:")
    for ct, count in counts.most_common(10):
        tag = "CORRECT" if ct == hw_ct.hex() else "FAULT"
        print(count, tag, ct)

for w, off in candidates:
    test_setting(w, off)


SETTING: w= 1 off= -3 ext= 0
Timeouts: 0
Correct: 114
Faulty: 386
Top 10:
114 CORRECT 69c4e0d86a7b0430d8cdb78070b4c55a
10 FAULT dc45a908a739e389b3b61333bf25bf71
9 FAULT dc45dd08a7b0e389b8b61333bf25bff7
5 FAULT 94713f2c36d3b301e51e0d96db073b41
5 FAULT d371c6a036d03d9551ad117de8503b64
4 FAULT 007119243611a613d36e0196d9203b7f
4 FAULT 94713ffd36d35901e5780d96c8073b41
4 FAULT d39d59a0e0e63d95e7ad11cce8508294
4 FAULT a371724e3635f9a1cf8b3060843d3b28
4 FAULT 724132ec544b3e6327bd07326f219345

SETTING: w= 1 off= 1 ext= 0
Timeouts: 0
Correct: 343
Faulty: 157
Top 10:
343 CORRECT 69c4e0d86a7b0430d8cdb78070b4c55a
2 FAULT e103358aa5e522d1a12dab32c56958f2
2 FAULT e6f4689c3e921faff488b33bfb87ef61
1 FAULT b10196970a0971e8e6306d9a5d649394
1 FAULT b1d0d497885271e9d9536d295d449e90
1 FAULT a7946b95b289705e8bbd67b2ecf2611c
1 FAULT 222a43fbd8699ee74e8d39ff98386f10
1 FAULT 610a3990bdd4e9a4dc9c6a4e23ff7543
1 FAULT e6a1c98ae10615af792379488a8f714b
1 FAULT f659934c780c71a01252bee05df7a4b0

SETTING: w= 3 off= -1

In [7]:
scope.arm()

target.fpga_write(target.REG_CRYPT_TEXTIN, PT)
target.fpga_write(target.REG_CRYPT_GO, b"\x01")

timeout = scope.capture()

print("timeout:", timeout)
print("trig_count:", scope.adc.trig_count)
print("adc_freq:", scope.clock.adc_freq)
print("clkgen_freq:", scope.clock.clkgen_freq)

trigger_time = scope.adc.trig_count / scope.clock.adc_freq
aes_cycles = trigger_time * scope.clock.clkgen_freq

print("trigger_time:", trigger_time)
print("approx AES cycles:", aes_cycles)

timeout: False
trig_count: 4
adc_freq: 29538459
clkgen_freq: 7384615.384615385
trigger_time: 1.3541667830403747e-07
approx AES cycles: 1.0000000859375076


In [ ]:
from collections import Counter
import time
import pandas as pd

GOLDEN = hw_ct.hex()

def diff_bytes(ct_hex, golden_hex=GOLDEN):
    ct = bytes.fromhex(ct_hex)
    golden = bytes.fromhex(golden_hex)
    return [i for i, (a, b) in enumerate(zip(ct, golden)) if a != b]

def test_setting(w, off, ext=0, n=500):
    scope.glitch.ext_offset = ext
    scope.glitch.width = w
    scope.glitch.offset = off
    scope.glitch.repeat = 1

    cts = []
    timeouts = 0

    for rep in range(n):
        scope.arm()

        target.fpga_write(target.REG_CRYPT_TEXTIN, PT)
        target.fpga_write(target.REG_CRYPT_GO, b"\x01")

        timeout = scope.capture()

        if timeout:
            timeouts += 1
            continue

        time.sleep(0.001)
        ct = bytes(target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16)).hex()
        cts.append(ct)

    counts = Counter(cts)

    faulty_counts = {ct: c for ct, c in counts.items() if ct != GOLDEN}

    if faulty_counts:
        top_fault_ct, top_fault_count = Counter(faulty_counts).most_common(1)[0]
        changed = diff_bytes(top_fault_ct)
    else:
        top_fault_ct = None
        top_fault_count = 0
        changed = []

    result = {
        "ext_offset": ext,
        "width": w,
        "offset": off,
        "trials": n,
        "valid_captures": len(cts),
        "timeouts": timeouts,
        "correct": counts[GOLDEN],
        "faulty": len(cts) - counts[GOLDEN],
        "top_fault_count": top_fault_count,
        "top_fault_ct": top_fault_ct,
        "changed_byte_count": len(changed),
        "changed_bytes": changed,
    }

    return result, counts


# Focus around your best point: w=-39, off=1, ext=0
results = []
all_counts = {}

ext_offsets = range(0, 10)
widths = range(-43, -34, 1)
offsets = range(-3, 6, 1)

N = 500

print("Starting focused repeatability scan...")

for ext in ext_offsets:
    for w in widths:
        for off in offsets:
            result, counts = test_setting(w, off, ext=ext, n=N)

            results.append(result)
            all_counts[(ext, w, off)] = counts

            if result["top_fault_count"] > 0:
                print(
                    f"ext={ext:2d} w={w:4d} off={off:3d} | "
                    f"correct={result['correct']:3d} faulty={result['faulty']:3d} "
                    f"top_fault={result['top_fault_count']:3d} "
                    f"changed_bytes={result['changed_byte_count']} "
                    f"ct={result['top_fault_ct']}"
                )

df = pd.DataFrame(results)

# Ranking: repeated fault first, then fewer changed bytes, then fewer timeouts
df_ranked = df.sort_values(
    by=["top_fault_count", "changed_byte_count", "timeouts"],
    ascending=[False, True, True]
)

print("\nTop candidate settings:")
print(df_ranked.head(20)[[
    "ext_offset",
    "width",
    "offset",
    "correct",
    "faulty",
    "timeouts",
    "top_fault_count",
    "changed_byte_count",
    "changed_bytes",
    "top_fault_ct"
]])

Starting focused repeatability scan...


(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work


ext= 0 w= -43 off=  1 | correct=496 faulty=  4 top_fault=  2 changed_bytes=16 ct=3e2e6da849e42fe48eba628df89ab6f4


(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work


ext= 0 w= -42 off=  1 | correct=498 faulty=  2 top_fault=  1 changed_bytes=16 ct=5daf05e2c1c819be6f7ddd69235d16d3


...

(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work


ext= 0 w= -40 off=  1 | correct=497 faulty=  3 top_fault=  1 changed_bytes=16 ct=3e9636a8de5718f44e7ddca5bc5d428c


(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work


ext= 0 w= -39 off=  1 | correct=494 faulty=  6 top_fault=  1 changed_bytes=14 ct=aa230d22bdbb552ed80d102a70f74e5e


(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work


In [14]:
best = df_ranked.iloc[0]

ext = int(best["ext_offset"])
w = int(best["width"])
off = int(best["offset"])

print("Best setting:", ext, w, off)

result, counts = test_setting(w, off, ext=ext, n=3000)

print(result)

print("\nTop 20 ciphertexts:")
for ct, count in counts.most_common(20):
    tag = "CORRECT" if ct == GOLDEN else "FAULT"
    print(count, tag, ct, diff_bytes(ct) if ct != GOLDEN else [])

Best setting: 1 -38 1
{'ext_offset': 1, 'width': -38, 'offset': 1, 'trials': 3000, 'valid_captures': 3000, 'timeouts': 0, 'correct': 2988, 'faulty': 12, 'top_fault_count': 4, 'top_fault_ct': '6dc405736a7b0430d8cd1b1ba1b4935a', 'changed_byte_count': 7, 'changed_bytes': [0, 2, 3, 10, 11, 12, 14]}

Top 20 ciphertexts:
2988 CORRECT 69c4e0d86a7b0430d8cdb78070b4c55a []
4 FAULT 6dc405736a7b0430d8cd1b1ba1b4935a [0, 2, 3, 10, 11, 12, 14]
2 FAULT 6dc4e0d86a7b0430d8cdb78070b4c55a [0]
1 FAULT 6dc418736a7b0430d8cd1b1ba1b4935a [0, 2, 3, 10, 11, 12, 14]
1 FAULT 2ef71aa7417fe3f7327dcb1b503864da [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
1 FAULT 007755f4df24c487324dcb949438e663 [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
1 FAULT 6dc4e0736a7b0430d8cdb71ba1b4bf5a [0, 3, 11, 12, 14]
1 FAULT 007755fedf01c4873229cb845038e4db [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
1 FAULT 007754f0df24c5918bba5396dd3ce663 [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]


In [15]:
from collections import Counter
import time

GOLDEN = hw_ct.hex()

candidates = [
    (1, -38, 1),   # ext, width, offset
    (0, -40, 1),
    (2, -39, 1),
]

N = 5000

def diff_bytes(ct_hex, golden_hex=GOLDEN):
    ct = bytes.fromhex(ct_hex)
    golden = bytes.fromhex(golden_hex)

    return [i for i, (a, b) in enumerate(zip(ct, golden)) if a != b]

for ext, w, off in candidates:

    print("\n" + "="*80)
    print(f"TESTING ext={ext}, width={w}, offset={off}")
    print("="*80)

    scope.glitch.ext_offset = ext
    scope.glitch.width = w
    scope.glitch.offset = off
    scope.glitch.repeat = 1

    cts = []
    timeouts = 0

    start = time.time()

    for i in range(N):

        scope.arm()

        target.fpga_write(target.REG_CRYPT_TEXTIN, PT)
        target.fpga_write(target.REG_CRYPT_GO, b"\x01")

        timeout = scope.capture()

        if timeout:
            timeouts += 1
            continue

        ct = bytes(target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16)).hex()
        cts.append(ct)

        if (i + 1) % 500 == 0:
            print(f"{i+1}/{N}")

    elapsed = time.time() - start

    counts = Counter(cts)

    correct = counts[GOLDEN]
    faulty = len(cts) - correct

    print("\nSUMMARY")
    print("Elapsed:", round(elapsed, 2), "seconds")
    print("Timeouts:", timeouts)
    print("Correct:", correct)
    print("Faulty :", faulty)

    print("\nTOP 20 CIPHERTEXTS")

    for ct, count in counts.most_common(20):

        if ct == GOLDEN:
            print(f"{count:5d}  CORRECT  {ct}")
        else:
            changed = diff_bytes(ct)

            print(
                f"{count:5d}  FAULT    {ct} "
                f"changed_bytes={len(changed)} "
                f"indices={changed}"
            )


TESTING ext=1, width=-38, offset=1
500/5000
1000/5000
1500/5000
2000/5000
2500/5000
3000/5000
3500/5000
4000/5000
4500/5000
5000/5000

SUMMARY
Elapsed: 15.76 seconds
Timeouts: 0
Correct: 4952
Faulty : 48

TOP 20 CIPHERTEXTS
 4952  CORRECT  69c4e0d86a7b0430d8cdb78070b4c55a
   14  FAULT    6dc4e0d86a7b0430d8cdb78070b4c55a changed_bytes=1 indices=[0]
    6  FAULT    6dc419736a7b0430d8cdb71ba1b4935a changed_bytes=6 indices=[0, 2, 3, 11, 12, 14]
    4  FAULT    6dc405736a7b0430d8cd1b1ba1b4935a changed_bytes=7 indices=[0, 2, 3, 10, 11, 12, 14]
    3  FAULT    007754f0cf24c5918bba5396dd3ce663 changed_bytes=16 indices=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
    2  FAULT    6cc418736a7b0430d9cd1b1ba0b4935a changed_bytes=8 indices=[0, 2, 3, 8, 10, 11, 12, 14]
    2  FAULT    007754f0df24c5918bba5396dd3ce663 changed_bytes=16 indices=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
    2  FAULT    6cc418736a7be330d9aa1b1ba083615a changed_bytes=11 indices=[0, 2, 3, 6, 8, 9, 10